# Library Import

In [1]:
# 한국어 텍스트 감정 분석을 위한 필수 라이브러리들
from collections import Counter  # 카운터 자료구조
import os  # 운영체제 인터페이스
import platform  # 플랫폼 정보
import re  # 정규 표현식
import sys  # 시스템 정보
import warnings  # 경고 메시지 제어


import matplotlib.pyplot as plt  # 데이터 시각화
plt.rc("font", family="NanumBarunGothic")  # 한글 폰트 설정(없으면 설치 필요)

import numpy as np  # 수치 연산
import pandas as pd  # 데이터 처리 및 분석
import seaborn as sns  # 고급 시각화
import torch  # 딥러닝 프레임워크
import koreanize_matplotlib

# 머신러닝 관련 라이브러리
from sklearn.metrics import accuracy_score, f1_score  # 평가 지표
from sklearn.model_selection import train_test_split  # 데이터 분할

# 트랜스포머 및 BERT 관련 라이브러리
from transformers import (
    AutoModelForSequenceClassification,  # 시퀀스 분류 모델
    AutoTokenizer,  # 토크나이저
    DataCollatorWithPadding,  # 패딩 데이터 콜레이터
    set_seed,
    Trainer,  # 트레이너
    TrainingArguments,  # 훈련 설정
)

# PyTorch 데이터 처리
from torch.utils.data import Dataset  # 데이터셋 및 데이터로더

# 경고 메시지 필터링
warnings.filterwarnings("ignore")

# 라이브러리 버전 정보 출력 (재현성을 위함)
print("=== 라이브러리 버전 정보 ===")
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"torch: {torch.__version__}")
print(f"transformers: {__import__('transformers').__version__}")
print(f"sklearn: {__import__('sklearn').__version__}")
print(f"matplotlib: {__import__('matplotlib').__version__}")
print(f"seaborn: {sns.__version__}")

# GPU 사용 가능 여부 확인
print("\n=== PyTorch GPU 지원 정보 ===")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 버전: {torch.version.cuda}")
    print(f"GPU 개수: {torch.cuda.device_count()}")
    print(f"현재 GPU: {torch.cuda.current_device()}")
    print(f"GPU 이름: {torch.cuda.get_device_name()}")
else:
    print("CPU에서 실행 중")

=== 라이브러리 버전 정보 ===
Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
pandas: 2.3.3
numpy: 2.3.3
torch: 2.9.0+cu128
transformers: 4.57.1
sklearn: 1.7.2
matplotlib: 3.10.7
seaborn: 0.13.2

=== PyTorch GPU 지원 정보 ===
CUDA 사용 가능: True
CUDA 버전: 12.8
GPU 개수: 1
현재 GPU: 0
GPU 이름: NVIDIA GeForce RTX 2070


## Random Seed Configuration

In [2]:
# 랜덤 시드 설정
RANDOM_STATE = 42


set_seed(RANDOM_STATE)

print(f"랜덤 시드 {RANDOM_STATE}로 설정 완료")

랜덤 시드 42로 설정 완료


# Data Load

In [3]:
# 데이터 로드
df = pd.read_csv("../../../data/raw/train.csv")

# 처음 몇 행 표시
print("\n처음 5행:")
df.head()


처음 5행:


,ID,review,label,type
0,0,이 영화는 정말 여성의 강인함과 힘을 제대로 보여주는 작품이었어요! 주인공이 자기 ...,2,augment
1,1,어느 부잣집 도련님의 철없는 행각,1,original
2,3,왜이렇게 재미가없냐 원도 별로였지만 원보다 더 재미없네,0,original
3,4,크리스마스 시즌엔 무조건 홈 알론이죠! 맥컬리 컬킨이 연기한 케빈의 재치있고 천방지...,2,augment
4,5,참나ㅋㅋ이게 무슨 드라마 최초 뮤지컬드라마야 이게무슨ㅋㅋ걍 다른 드라마랑 똑같구만 ...,0,original


In [4]:
# ID 1269인 데이터 확인
df_1269 = df[df['ID'] == 1269]
print(f"\nID 1269 데이터:")
pd.set_option('display.max_colwidth', None)
print(df_1269)
pd.reset_option('display.max_colwidth')



ID 1269 데이터:
        ID  \
1043  1269   

                                                                                                                                                                                                                                                                           review  \
1043  고질라 영화 재밌게 볼 수 있는 꿀팁이 있어서 공유합니다! 이 링크를 타고 가면 고질라를 더욱 흥미진진하게 관람할 수 있는 비법이 적혀있어요. http://movie.naver.com/movie/bi/mi/reviewread.nhn?code=100987&nid=3431254#tab 여기 가서 한 번 읽어보시면 엄청난 도움이 될 거예요. 그리고 꼭 추천하기 버튼을 눌러주세요! 여러분의 추천이 더 많은 사람들이 이 꿀팁을 알게 해줄 거예요. 함께 고질라를 더 재밌게 봐요! 🐲🔥   

      label     type  
1043      2  augment  


# Data/Feature Engineering

In [5]:
df_processed = df[["ID", "label", "review", "type"]].copy()

print(f"원본 데이터셋 크기: {len(df_processed):,}개")

df_processed.head()

원본 데이터셋 크기: 279,650개


,ID,label,review,type
0,0,2,이 영화는 정말 여성의 강인함과 힘을 제대로 보여주는 작품이었어요! 주인공이 자기 ...,augment
1,1,1,어느 부잣집 도련님의 철없는 행각,original
2,3,0,왜이렇게 재미가없냐 원도 별로였지만 원보다 더 재미없네,original
3,4,2,크리스마스 시즌엔 무조건 홈 알론이죠! 맥컬리 컬킨이 연기한 케빈의 재치있고 천방지...,augment
4,5,0,참나ㅋㅋ이게 무슨 드라마 최초 뮤지컬드라마야 이게무슨ㅋㅋ걍 다른 드라마랑 똑같구만 ...,original


## 텍스트 정규화

- 대소문자 정규화 (BERT가 처리하지만)
- 구두점 처리
- 특수문자 처리
- URL/이메일/멘션 정리 (있는 경우)

In [28]:
print("텍스트 정규화 과정 시작")
print("=" * 50)

# 대소문자 정규화 (BERT가 처리하지만 일관성을 위해)
print("\n1단계: 대소문자 정규화 수행 중...")
df_processed["review_normalized"] = df_processed["review"].str.lower()
print("✓ 소문자 변환 완료")

# 구두점 정규화
print("\n2단계: 구두점 정규화 수행 중...")


def normalize_punctuation(text):
    # NaN 또는 float 타입 체크
    if pd.isna(text) or not isinstance(text, str):
        return text
    
    # 여러 개의 구두점을 하나로 정규화
    text = re.sub(r"[.]{2,}", ".", text)
    text = re.sub(r"[!]{2,}", "!", text)
    text = re.sub(r"[?]{2,}", "?", text)
    text = re.sub(r"[,]{2,}", ",", text)

    # 구두점 주변 공백 정리
    text = re.sub(r"\s+([.,!?])", r"\1", text)
    text = re.sub(r"([.,!?])\s+", r"\1 ", text)

    return text


df_processed["review_normalized"] = df_processed["review_normalized"].apply(
    normalize_punctuation
)
print("✓ 구두점 정규화 완료")

# 특수문자 추가 정리
print("\n3단계: 특수문자 정리 수행 중...")


def clean_special_chars(text):
    # NaN 또는 float 타입 체크
    if pd.isna(text) or not isinstance(text, str):
        return text
    
    # URL 패턴 제거 (있는 경우)
    # URL 패턴 제거 (http/https 및 fragment 포함)
    text = re.sub(
        r"http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+#]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+",
        "",
        text,
    )
    text = re.sub(
        r"www\.[a-zA-Z0-9\-_~:/?#\[\]@!$&'()*+,;=.]+",
        "",
        text,
    )

    # '노려p://'와 같은 케이스에서 p://만 사라지게 하려면 바로 뒤에 오는 p:// 패턴만 지우는 것이 적합합니다.
    # 기존 정규식은 p:// 뒤에 반드시 '.'이 온다고 가정하여 오작동하며, 문자열을 너무 많이 지웁니다.
    # 아래처럼 p://만 삭제하는 쪽이 의도와 부합합니다.
    text = re.sub(r"p://", "", text)

    # 지정된 도메인 리스트 확장에 따라 도메인 패턴 제거
    tlds = [
        'com', 'net', 'org', 'co', 'kr', 'io', 'me', 'info', 'biz', 'tv', 'ai', 'app', 'dev',
        'xyz', 'us', 'uk', 'jp', 'cn', 'ru', 'site', 'store', 'online', 'top', 'tech', 'shop', 'cloud'
    ]
    tld_pattern = "|".join(tlds)
    # 'something.tld' 또는 'something.something.tld' 등, 최소한 도메인 알파벳이 마지막인 경우
    text = re.sub(
        rf"\b[a-zA-Z0-9\-_]+(?:\.[a-zA-Z0-9\-_]+)*\.({tld_pattern})\b",
        "", text)
    


    # 이메일 패턴 제거 (있는 경우)
    text = re.sub(r"\S+@\S+", "", text)

    # sweeti-potato@와 같이 '@'로 끝나는 이메일 fragment 제거
    text = re.sub(r"\b[\w\.\-]+@\b", "", text)
    # 이메일 username@ 처럼 @로 끝나는 형태 - username까지 같이 삭제
    text = re.sub(r"\b[\w\.\-]+@(?=\s|$)", "", text)

    # hsj1549@ 처럼 '@'로 끝나는 멘션 등도 한 번 더 처리
    text = re.sub(r"@\b", "", text)
    text = re.sub(r"\b@\b", "", text)
    text = re.sub(r"\b@\s", " ", text)

    # 멘션 패턴 제거 (있는 경우)
    text = re.sub(r"@\w+", "", text)

    # 과도한 공백 정리
    text = re.sub(r"\s+", " ", text)


    # 『내용』, 【내용】, 《내용》, ｢내용｣ 등 특수 괄호(전각 포함)로 둘러싸인 텍스트 제거 (여러 번 등장 가능하므로 반복적으로 모두 제거)
    # 대표적인 특수 괄호 모음: 『 』 【 】 《 》 「 」 〈 〉 ｢ ｣ “ ” ‘ ’
    # re.DOTALL 옵션으로 줄바꿈 포함 영역 지움, 반복 적용해서 모든 패턴 제거
    special_bracket_pattern = r"[『【《「〈｢“‘](.*?)[』】》」〉｣”’]"
    prev_text = None
    while prev_text != text:
        prev_text = text
        text = re.sub(special_bracket_pattern, "", text, flags=re.DOTALL)

    # 날짜 제거 (YYYY-MM-DD, YYYY/MM/DD, YY.MM.DD, YYYY.MM.DD, YYYY년MM월DD일, MM/DD, MM-DD, MM.DD 등)
    date_patterns = [
        r'\b\d{4}[-/.]\d{1,2}[-/.]\d{1,2}\b',      # 2021-12-30, 2021/12/30, 2021.12.30
        r'\b\d{2}[-/.]\d{1,2}[-/.]\d{1,2}\b',      # 21-12-30, 21.12.30, 21/12/30
        r'\b\d{1,2}[-/.]\d{1,2}\b',                # 12-30, 12.30, 12/30 (월일)
        r'\b\d{4}년\s?\d{1,2}월\s?\d{1,2}일\b',     # 2021년 12월 30일, 2021년12월30일
        r'\b\d{2}년\s?\d{1,2}월\s?\d{1,2}일\b',     # 21년 12월 30일
        r'\b\d{4}년\s?\d{1,2}월\b',                # 2021년 12월 (년월)
        r'\b\d{4}년\b',                            # 2021년
    ]
    for pat in date_patterns:
        text = re.sub(pat, "", text)

    # 전화번호 제거 패턴 추가
    phone_patterns = [
        r'\b01[016789][ -]?\d{3,4}[ -]?\d{4}\b',        # 010-1234-5678, 011 222 3333, 01612345678 등
        r'\b\d{2,4}[ -]?\d{3,4}[ -]?\d{4}\b',           # 02-123-4567, 053 123 4567, 0311234567 등 일반 번호
        r'\b\d{4}[ -]?\d{4}\b',                         # 1234-5678, 12345678 등
    ]
    for pat in phone_patterns:
        text = re.sub(pat, "", text)

    # 금액 패턴 제거
    price_patterns = [
        r'\b\d+원\b',                    # 1000원, 50000원
        r'\b\d+,\d+원\b',               # 1,000원, 50,000원
        r'\b\d+\.\d+원\b',              # 1000.5원
        r'\b\d+만원\b',                 # 1만원, 10만원
        r'\b\d+천원\b',                 # 1천원, 5천원
        r'\b\d+억원\b',                 # 1억원
        r'\$\d+',                       # $100, $50
        r'\b\d+달러\b',                 # 100달러
    ]
    for pat in price_patterns:
        text = re.sub(pat, "", text)

    # 시간 패턴 제거
    time_patterns = [
        r'\b\d{1,2}:\d{2}(?::\d{2})?\b',  # 14:30, 14:30:25
        r'\b\d{1,2}시\s?\d{1,2}분\b',     # 2시 30분, 2시30분
        r'\b\d{1,2}시간\b',               # 2시간, 10시간
        r'\b\d{1,2}분\b',                 # 30분, 5분
        r'\b\d{1,2}초\b',                 # 30초, 5초
        r'\b오전\s?\d{1,2}시\b',          # 오전 9시
        r'\b오후\s?\d{1,2}시\b',          # 오후 2시
    ]  
    for pat in time_patterns:
        text = re.sub(pat, "", text)

    # 영화 관련 패턴 제거
    movie_patterns = [
        r'\b\d+편\b',                    # 1편, 2편, 3편
        r'\b\d+부작\b',                  # 1부작, 2부작
        r'\b\d+기\b',                    # 1기, 2기
        r'\b\d+회차\b',                  # 1회차, 2회차
        r'\b\d+화\b',                    # 1화, 2화
        r'\b\d+분\s?\d+초\b',            # 120분 30초
        r'\b\d+분\b',                    # 120분 (영화 상영시간)
        r'\b\d+등급\b',                  # 15등급, 18등급
        r'\b\d+세\s?이상\b',             # 15세 이상
    ]    
    for pat in movie_patterns:
        text = re.sub(pat, "", text)

    # SNS/플랫폼 패턴 제거
    sns_patterns = [
        r'\b#\w+\b',                     # 해시태그 #영화 #추천
        r'\b@\w+\b',                     # 멘션 @username
        r'\bRT\b',                       # 리트윗 표시
        r'\b좋아요\s?\d+\b',             # 좋아요 100
        r'\b댓글\s?\d+\b',               # 댓글 50
        r'\b공유\s?\d+\b',               # 공유 20
        r'\b조회수\s?\d+\b',             # 조회수 1000
        r'\b구독자\s?\d+\b',             # 구독자 5000
    ]
    for pat in sns_patterns:
        text = re.sub(pat, "", text)

    # 기타 노이즈 패턴 제거
    noise_patterns = [
        r'\b\d+번\b',                    # 1번, 2번
        r'\b\d+개\b',                    # 1개, 2개
        r'\b\d+명\b',                    # 1명, 2명
        r'\b\d+장\b',                    # 1장, 2장
        r'\b\d+회\b',                    # 1회, 2회
        r'\b\d+차\b',                    # 1차, 2차
        r'\b\d+번째\b',                  # 1번째, 2번째
        r'\b\d+위\b',                    # 1위, 2위
        r'\b\d+등\b',                    # 1등, 2등
        r'\b\d+점\b',                    # 1점, 2점
        r'\b\d+점대\b',                  # 1점대, 2점대
        r'\b\d+점만점\b',                # 10점만점
        r'\b\d+점\s?만점\b',             # 10점 만점
    ]
    for pat in noise_patterns:
        text = re.sub(pat, "", text)

    # 특수 문자 및 기호 제거
    special_chars = [
        r'[★☆♥♡♠♣♦]',                  # 특수 기호
        r'[♪♫♬♩]',                      # 음악 기호
        r'[→←↑↓]',                      # 화살표
        r'[①②③④⑤⑥⑦⑧⑨⑩]',            # 원 숫자
        r'[⑴⑵⑶⑷⑸⑹⑺⑻⑼⑽]',            # 괄호 숫자
        r'[❶❷❸❹❺❻❼❽❾❿]',            # 검은 원 숫자
        r'[ⓐⓑⓒⓓⓔⓕⓖⓗⓘⓙ]',            # 원 문자
    ]
    for pat in special_chars:
        text = re.sub(pat, "", text)

    return text.strip()


df_processed["review_normalized"] = df_processed["review_normalized"].apply(
    clean_special_chars
)
print("✓ URL/이메일/멘션 제거 완료")

# 정규화 후 빈 텍스트 처리
print("\n4단계: 정규화 후 빈 텍스트 확인 중...")
empty_after_normalization = df_processed["review_normalized"].str.strip().eq("").sum()
if empty_after_normalization > 0:
    df_processed = df_processed[df_processed["review_normalized"].str.strip() != ""]
    print(f"✓ 빈 텍스트 {empty_after_normalization}개 제거")
else:
    print("✓ 빈 텍스트 없음")

# 정규화 전후 비교
print("\n" + "=" * 50)
print("정규화 결과 요약:")
print(f"최종 데이터 크기: {len(df_processed):,}개")
print(
    f"평균 길이 - 정규화됨: {df_processed['review_normalized'].str.len().mean():.1f}자"
)
print("=" * 50)

텍스트 정규화 과정 시작

1단계: 대소문자 정규화 수행 중...
✓ 소문자 변환 완료

2단계: 구두점 정규화 수행 중...
✓ 구두점 정규화 완료

3단계: 특수문자 정리 수행 중...
✓ URL/이메일/멘션 제거 완료

4단계: 정규화 후 빈 텍스트 확인 중...
✓ 빈 텍스트 110개 제거

정규화 결과 요약:
최종 데이터 크기: 279,515개
평균 길이 - 정규화됨: 157.6자


In [8]:
# 데이터 확인
df_processed.head()

,ID,label,review,type,review_normalized
0,0,2,이 영화는 정말 여성의 강인함과 힘을 제대로 보여주는 작품이었어요! 주인공이 자기 ...,augment,이 영화는 정말 여성의 강인함과 힘을 제대로 보여주는 작품이었어요! 주인공이 자기 ...
1,1,1,어느 부잣집 도련님의 철없는 행각,original,어느 부잣집 도련님의 철없는 행각
2,3,0,왜이렇게 재미가없냐 원도 별로였지만 원보다 더 재미없네,original,왜이렇게 재미가없냐 원도 별로였지만 원보다 더 재미없네
3,4,2,크리스마스 시즌엔 무조건 홈 알론이죠! 맥컬리 컬킨이 연기한 케빈의 재치있고 천방지...,augment,크리스마스 시즌엔 무조건 홈 알론이죠! 맥컬리 컬킨이 연기한 케빈의 재치있고 천방지...
4,5,0,참나ㅋㅋ이게 무슨 드라마 최초 뮤지컬드라마야 이게무슨ㅋㅋ걍 다른 드라마랑 똑같구만 ...,original,참나ㅋㅋ이게 무슨 드라마 최초 뮤지컬드라마야 이게무슨ㅋㅋ걍 다른 드라마랑 똑같구만 ...


In [27]:
# 정규화 전후 비교 - 변경된 항목만 출력 및 파일 저장
print("정규화 전후 비교 (변경된 항목만)")
print("=" * 50)

# review와 review_normalized가 다른 항목 필터링
changed_normalized = df_processed[df_processed["review"] != df_processed["review_normalized"]]

if len(changed_normalized) > 0:
    print(f"\n변경된 항목 수: {len(changed_normalized):,}개")
    
    # augment와 original 타입으로 분리
    changed_augment = changed_normalized[changed_normalized["type"] == "augment"]
    changed_original = changed_normalized[changed_normalized["type"] == "original"]
    
    print(f"  - augment 타입: {len(changed_augment):,}개")
    print(f"  - original 타입: {len(changed_original):,}개")
    
    # augment 타입 파일 저장
    with open("normalization_comparison_augment.txt", "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write("정규화 전후 비교 - AUGMENT 타입 (변경된 항목만)\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"총 변경된 항목 수: {len(changed_augment):,}개\n")
        f.write("=" * 80 + "\n\n")
        
        for idx, row in changed_augment.iterrows():
            f.write(f"{'─' * 80}\n")
            f.write(f"ID: {row['ID']}\n")
            f.write(f"Label: {row['label']}\n")
            f.write(f"{'─' * 80}\n")
            f.write(f"[정리된 리뷰]\n{row['review']}\n\n")
            f.write(f"[정규화된 리뷰]\n{row['review_normalized']}\n")
            f.write(f"{'─' * 80}\n\n")
    
    print(f"\n✓ augment 타입 비교 결과 저장: normalization_comparison_augment.txt")
    
    # original 타입 파일 저장
    with open("normalization_comparison_original.txt", "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write("정규화 전후 비교 - ORIGINAL 타입 (변경된 항목만)\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"총 변경된 항목 수: {len(changed_original):,}개\n")
        f.write("=" * 80 + "\n\n")
        
        for idx, row in changed_original.iterrows():
            f.write(f"{'─' * 80}\n")
            f.write(f"ID: {row['ID']}\n")
            f.write(f"Label: {row['label']}\n")
            f.write(f"{'─' * 80}\n")
            f.write(f"[정리된 리뷰]\n{row['review']}\n\n")
            f.write(f"[정규화된 리뷰]\n{row['review_normalized']}\n")
            f.write(f"{'─' * 80}\n\n")
    
    print(f"✓ original 타입 비교 결과 저장: normalization_comparison_original.txt")
    
    # 콘솔에 샘플 출력
    print("\n샘플 출력 (각 타입별 최대 5개):")
    print("\n[AUGMENT 타입 샘플]")
    print("-" * 50)
    for idx, row in changed_augment.head(5).iterrows():
        print(f"\n[ID: {row['ID']}]")
        print(f"정리됨: {row['review']}")
        print(f"정규화: {row['review_normalized']}")
        print("-" * 50)
    
    print("\n[ORIGINAL 타입 샘플]")
    print("-" * 50)
    for idx, row in changed_original.head(5).iterrows():
        print(f"\n[ID: {row['ID']}]")
        print(f"정리됨: {row['review']}")
        print(f"정규화: {row['review_normalized']}")
        print("-" * 50)
else:
    print("\n변경된 항목이 없습니다.")

print("=" * 50)


정규화 전후 비교 (변경된 항목만)

변경된 항목 수: 115,158개
  - augment 타입: 64,291개
  - original 타입: 50,867개

✓ augment 타입 비교 결과 저장: normalization_comparison_augment.txt
✓ original 타입 비교 결과 저장: normalization_comparison_original.txt

샘플 출력 (각 타입별 최대 5개):

[AUGMENT 타입 샘플]
--------------------------------------------------

[ID: 9]
정리됨: 진짜 이 영화 보다보면 내가 다 암 걸릴 듯. 등장인물들 하나같이 다 정신없고, 짜증나는 전개에 머리가 지끈지끈거려. 도대체 무슨 생각으로 이렇게 만들어놨는지 감독이 궁금할 정도야. 중간중간 이해할 수 없는 행동들에 그냥 멍~하게 보고만 있었음. 이 영화 본 걸 생각하면 지금도 짜증이 막 나. 도대체 누가 이런 시나리오를 냈는지... 진짜 보는 내내 한숨밖에 안 나왔어. 이런 영화는 그냥 피하는 게 답인 듯. 평점 믿고 봤다가 시간만 버렸어. ㅠㅠ
정규화: 진짜 이 영화 보다보면 내가 다 암 걸릴 듯. 등장인물들 하나같이 다 정신없고, 짜증나는 전개에 머리가 지끈지끈거려. 도대체 무슨 생각으로 이렇게 만들어놨는지 감독이 궁금할 정도야. 중간중간 이해할 수 없는 행동들에 그냥 멍~하게 보고만 있었음. 이 영화 본 걸 생각하면 지금도 짜증이 막 나. 도대체 누가 이런 시나리오를 냈는지. 진짜 보는 내내 한숨밖에 안 나왔어. 이런 영화는 그냥 피하는 게 답인 듯. 평점 믿고 봤다가 시간만 버렸어. ㅠㅠ
--------------------------------------------------

[ID: 11]
정리됨: 와... 진짜 제대로 짱입니다 ㅡㅡ! 이 영화 하나로 한 주를 버틸 수 있을 것 같아요. 처음부터 끝까지 눈을 뗄 수가 없었어요. 특히 액션 장면은 진짜 소름 돋을 정도로 멋졌어요. 배우들의 연기

## 텍스트 전처리 및 정리

- 기본 텍스트 정리 (특수문자 제거, 공백 정규화)
- 필요시 한국어 특화 전처리 처리
- 매우 짧거나 긴 텍스트 제거 또는 처리
- 중복 제거

In [29]:
# 기본 텍스트 정리 함수
def clean_text(text):
    """
    한국어 텍스트를 위한 기본 텍스트 정리 함수

    전처리 단계:
    1. 불완전한 한글 제거 (자음/모음만 있는 경우)
    2. 반복되는 감정 표현 정규화 (ㅋㅋㅋ, ㅠㅠㅠ 등)
    3. 과도한 문자 반복 축소 (4번 이상 → 3번으로)
    4. 특수문자 제거 (한글, 숫자, 기본 구두점, 감정표현 제외)
    5. 공백 정규화
    """
    if pd.isna(text):
        return ""

    text = str(text).strip()

    # 한국어 특화 전처리
    text = re.sub(
        r"[ㄱ-ㅎㅏ-ㅣ]+", "", text
    )  # 불완전한 한글 제거 (자음/모음만 있는 경우)
    text = re.sub(r"([ㅋㅎ])\1{2,}", r"\1\1", text)  # 웃음 표현 정규화: ㅋㅋㅋ+ → ㅋㅋ
    text = re.sub(
        r"([ㅠㅜㅡ])\1{2,}", r"\1\1", text
    )  # 슬픔 표현 정규화: ㅠㅠㅠ+ → ㅠㅠ
    text = re.sub(
        r"(.)\1{3,}", r"\1\1\1", text
    )  # 과도한 반복 축소: 4번 이상 반복 → 3번으로
    text = re.sub(r"[^\w\s가-힣.,!?ㅋㅎㅠㅜㅡ~\-]", " ", text)  # 필요한 문자만 유지
    text = re.sub(r"\s+", " ", text)  # 다중 공백을 단일 공백으로

    return text.strip()


print("텍스트 전처리 과정 시작")
print("=" * 50)
initial_size = len(df_processed)
print(f"초기 데이터 크기: {initial_size:,}개")

# 1단계: 텍스트 정리 적용
print("\n1단계: 기본 텍스트 정리 수행 중...")
df_processed["review_cleaned"] = df_processed["review_normalized"].apply(clean_text)
print("✓ 한글 자음/모음 정리, 반복 표현 정규화, 특수문자 제거 완료")

# 2단계: 빈 텍스트 제거
print("\n2단계: 빈 텍스트 제거 중...")
empty_count = df_processed["review_cleaned"].str.strip().eq("").sum()
if empty_count > 0:
    df_processed = df_processed[df_processed["review_cleaned"].str.strip() != ""]
    print(f"✓ 빈 텍스트 {empty_count}개 제거")
else:
    print("✓ 빈 텍스트 없음")

# 3단계: 중복 제거
print("\n3단계: 중복 데이터 제거 중...")
duplicates_count = df_processed.duplicated(subset=["review_cleaned", "label"]).sum()
if duplicates_count > 0:
    df_processed = df_processed.drop_duplicates(subset=["review_cleaned", "label"])
    print(f"✓ 중복 데이터 {duplicates_count}개 제거")
else:
    print("✓ 중복 데이터 없음")

# 최종 결과 요약
final_size = len(df_processed)
removed = initial_size - final_size
print("\n" + "=" * 50)
print("전처리 결과 요약:")
print(f"최종 데이터 크기: {initial_size:,} → {final_size:,}")
print(f"제거된 데이터: {removed:,}개 ({removed / initial_size * 100:.1f}%)")
print("=" * 50)

텍스트 전처리 과정 시작
초기 데이터 크기: 279,515개

1단계: 기본 텍스트 정리 수행 중...
✓ 한글 자음/모음 정리, 반복 표현 정규화, 특수문자 제거 완료

2단계: 빈 텍스트 제거 중...
✓ 빈 텍스트 350개 제거

3단계: 중복 데이터 제거 중...
✓ 중복 데이터 3898개 제거

전처리 결과 요약:
최종 데이터 크기: 279,515 → 275,267
제거된 데이터: 4,248개 (1.5%)


In [30]:
# review_normalized와 review_cleaned 비교로 작성
print("review_normalized vs review_cleaned 비교 (변경된 항목만)")
print("=" * 50)

# review_normalized와 review_cleaned가 다른 항목 필터링
changed_reviews = df_processed[df_processed["review_normalized"] != df_processed["review_cleaned"]]

if len(changed_reviews) > 0:
    print(f"\n변경된 항목 수: {len(changed_reviews):,}개")
    
    # augment와 original 타입으로 분리
    changed_augment = changed_reviews[changed_reviews["type"] == "augment"]
    changed_original = changed_reviews[changed_reviews["type"] == "original"]
    
    print(f"  - augment 타입: {len(changed_augment):,}개")
    print(f"  - original 타입: {len(changed_original):,}개")
    
    # augment 타입 파일 저장
    with open("normalization_comparison_augment.txt", "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write("정규화 vs 정리 - AUGMENT 타입 (변경된 항목만)\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"총 변경된 항목 수: {len(changed_augment):,}개\n")
        f.write("=" * 80 + "\n\n")
        for idx, row in changed_augment.iterrows():
            f.write(f"{'─' * 80}\n")
            f.write(f"ID: {row['ID']}\n")
            f.write(f"Label: {row['label']}\n")
            f.write(f"{'─' * 80}\n")
            f.write(f"[정규화된 리뷰]\n{row['review_normalized']}\n\n")
            f.write(f"[정리된 리뷰]\n{row['review_cleaned']}\n")
            f.write(f"{'─' * 80}\n\n")
    print(f"\n✓ augment 타입 비교 결과 저장: normalization_comparison_augment.txt")

    # original 타입 파일 저장
    with open("normalization_comparison_original.txt", "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write("정규화 vs 정리 - ORIGINAL 타입 (변경된 항목만)\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"총 변경된 항목 수: {len(changed_original):,}개\n")
        f.write("=" * 80 + "\n\n")
        for idx, row in changed_original.iterrows():
            f.write(f"{'─' * 80}\n")
            f.write(f"ID: {row['ID']}\n")
            f.write(f"Label: {row['label']}\n")
            f.write(f"{'─' * 80}\n")
            f.write(f"[정규화된 리뷰]\n{row['review_normalized']}\n\n")
            f.write(f"[정리된 리뷰]\n{row['review_cleaned']}\n")
            f.write(f"{'─' * 80}\n\n")
    print(f"✓ original 타입 비교 결과 저장: normalization_comparison_original.txt")

    # 콘솔에 샘플 출력
    print("\n샘플 출력 (각 타입별 최대 5개):")
    print("\n[AUGMENT 타입 샘플]")
    print("-" * 50)
    for idx, row in changed_augment.head(5).iterrows():
        print(f"\n[ID: {row['ID']}]")
        print(f"정규화: {row['review_normalized']}")
        print(f"정리: {row['review_cleaned']}")
        print("-" * 50)
    
    print("\n[ORIGINAL 타입 샘플]")
    print("-" * 50)
    for idx, row in changed_original.head(5).iterrows():
        print(f"\n[ID: {row['ID']}]")
        print(f"정규화: {row['review_normalized']}")
        print(f"정리: {row['review_cleaned']}")
        print("-" * 50)
else:
    print("\n변경된 항목이 없습니다.")

print("=" * 50)


review_normalized vs review_cleaned 비교 (변경된 항목만)

변경된 항목 수: 119,403개
  - augment 타입: 89,119개
  - original 타입: 30,284개

✓ augment 타입 비교 결과 저장: normalization_comparison_augment.txt
✓ original 타입 비교 결과 저장: normalization_comparison_original.txt

샘플 출력 (각 타입별 최대 5개):

[AUGMENT 타입 샘플]
--------------------------------------------------

[ID: 4]
정규화: 크리스마스 시즌엔 무조건 홈 알론이죠! 맥컬리 컬킨이 연기한 케빈의 재치있고 천방지축인 모습을 보면서 웃음꽃이 피지 않은 이가 없을 거예요. 특히 그 두 명의 도둑들과의 에피소드들은 정말 빵빵 터지고, 보면서 손에 땀을 쥐게 만드는 긴장감도 최고였죠. 정말 크리스마스의 클래식이라고 할 수 있는 이 영화를 보면서 가족들과 함께 따뜻한 시간을 보내는 건 어떨까요? 영화가 끝나면 모두가 "메리 크리스마스"를 외치며 행복한 기분에 젖어 들 거예요! 🎅🏼🎁
정리: 크리스마스 시즌엔 무조건 홈 알론이죠! 맥컬리 컬킨이 연기한 케빈의 재치있고 천방지축인 모습을 보면서 웃음꽃이 피지 않은 이가 없을 거예요. 특히 그 두 명의 도둑들과의 에피소드들은 정말 빵빵 터지고, 보면서 손에 땀을 쥐게 만드는 긴장감도 최고였죠. 정말 크리스마스의 클래식이라고 할 수 있는 이 영화를 보면서 가족들과 함께 따뜻한 시간을 보내는 건 어떨까요? 영화가 끝나면 모두가 메리 크리스마스 를 외치며 행복한 기분에 젖어 들 거예요!
--------------------------------------------------

[ID: 9]
정규화: 진짜 이 영화 보다보면 내가 다 암 걸릴 듯. 등장인물들 하나같이 다 정신없고, 짜증나는 전개에 머리가 지끈지끈거려. 도대체 무슨 생각으로 이렇게 만들어

In [ ]:
class TextPreprocessingPipeline:
    def __init__(self):
        self.is_fitted = False
        self.vocab_info = {}
        self.label_patterns = {}
        # text_preprocessing.ipynb에서 개발된 추가 전처리 규칙 저장
        self.advanced_rules = {}  

    def basic_preprocess(self, texts):
        """기본 전처리 (clean_text + normalize 기능)"""
        processed_texts = []
        for text in texts:
            # 기본 텍스트 정리
            cleaned = self._clean_text(text)
            # 정규화
            normalized = self._normalize_text(cleaned)
            processed_texts.append(normalized)
        return processed_texts

    def advanced_preprocess(self, texts):
        """text_preprocessing.ipynb의 고급 전처리 적용"""
        processed_texts = []
        for text in texts:
            # 올바른 전처리 순서로 한 번만 처리
            text = self._normalize_punctuation(text)
            text = self._clean_special_chars(text)
            text = self._clean_text(text)
            processed_texts.append(text)
        return processed_texts


    def _normalize_punctuation(self, text):
        # NaN 또는 float 타입 체크
        if pd.isna(text) or not isinstance(text, str):
            return text
        
        # 여러 개의 구두점을 하나로 정규화
        text = re.sub(r"[.]{2,}", ".", text)
        text = re.sub(r"[!]{2,}", "!", text)
        text = re.sub(r"[?]{2,}", "?", text)
        text = re.sub(r"[,]{2,}", ",", text)

        # 구두점 주변 공백 정리
        text = re.sub(r"\s+([.,!?])", r"\1", text)
        text = re.sub(r"([.,!?])\s+", r"\1 ", text)

        return text

    def _clean_special_chars(self, text):
        # NaN 또는 float 타입 체크
        if pd.isna(text) or not isinstance(text, str):
            return text
        
        # URL 패턴 제거 (있는 경우)
        # URL 패턴 제거 (http/https 및 fragment 포함)
        text = re.sub(
            r"http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+#]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+",
            "",
            text,
        )
        text = re.sub(
            r"www\.[a-zA-Z0-9\-_~:/?#\[\]@!$&'()*+,;=.]+",
            "",
            text,
        )

        # '노려p://'와 같은 케이스에서 p://만 사라지게 하려면 바로 뒤에 오는 p:// 패턴만 지우는 것이 적합합니다.
        # 기존 정규식은 p:// 뒤에 반드시 '.'이 온다고 가정하여 오작동하며, 문자열을 너무 많이 지웁니다.
        # 아래처럼 p://만 삭제하는 쪽이 의도와 부합합니다.
        text = re.sub(r"p://", "", text)

        # 지정된 도메인 리스트 확장에 따라 도메인 패턴 제거
        tlds = [
            'com', 'net', 'org', 'co', 'kr', 'io', 'me', 'info', 'biz', 'tv', 'ai', 'app', 'dev',
            'xyz', 'us', 'uk', 'jp', 'cn', 'ru', 'site', 'store', 'online', 'top', 'tech', 'shop', 'cloud'
        ]
        tld_pattern = "|".join(tlds)
        # 'something.tld' 또는 'something.something.tld' 등, 최소한 도메인 알파벳이 마지막인 경우
        text = re.sub(
            rf"\b[a-zA-Z0-9\-_]+(?:\.[a-zA-Z0-9\-_]+)*\.({tld_pattern})\b",
            "", text)
        


        # 이메일 패턴 제거 (있는 경우)
        text = re.sub(r"\S+@\S+", "", text)

        # sweeti-potato@와 같이 '@'로 끝나는 이메일 fragment 제거
        text = re.sub(r"\b[\w\.\-]+@\b", "", text)
        # 이메일 username@ 처럼 @로 끝나는 형태 - username까지 같이 삭제
        text = re.sub(r"\b[\w\.\-]+@(?=\s|$)", "", text)

        # hsj1549@ 처럼 '@'로 끝나는 멘션 등도 한 번 더 처리
        text = re.sub(r"@\b", "", text)
        text = re.sub(r"\b@\b", "", text)
        text = re.sub(r"\b@\s", " ", text)

        # 멘션 패턴 제거 (있는 경우)
        text = re.sub(r"@\w+", "", text)

        # 과도한 공백 정리
        text = re.sub(r"\s+", " ", text)


        # 『내용』, 【내용】, 《내용》, ｢내용｣ 등 특수 괄호(전각 포함)로 둘러싸인 텍스트 제거 (여러 번 등장 가능하므로 반복적으로 모두 제거)
        # 대표적인 특수 괄호 모음: 『 』 【 】 《 》 「 」 〈 〉 ｢ ｣ “ ” ‘ ’
        # re.DOTALL 옵션으로 줄바꿈 포함 영역 지움, 반복 적용해서 모든 패턴 제거
        special_bracket_pattern = r"[『【《「〈｢“‘](.*?)[』】》」〉｣”’]"
        prev_text = None
        while prev_text != text:
            prev_text = text
            text = re.sub(special_bracket_pattern, "", text, flags=re.DOTALL)

        # 날짜 제거 (YYYY-MM-DD, YYYY/MM/DD, YY.MM.DD, YYYY.MM.DD, YYYY년MM월DD일, MM/DD, MM-DD, MM.DD 등)
        date_patterns = [
            r'\b\d{4}[-/.]\d{1,2}[-/.]\d{1,2}\b',      # 2021-12-30, 2021/12/30, 2021.12.30
            r'\b\d{2}[-/.]\d{1,2}[-/.]\d{1,2}\b',      # 21-12-30, 21.12.30, 21/12/30
            r'\b\d{1,2}[-/.]\d{1,2}\b',                # 12-30, 12.30, 12/30 (월일)
            r'\b\d{4}년\s?\d{1,2}월\s?\d{1,2}일\b',     # 2021년 12월 30일, 2021년12월30일
            r'\b\d{2}년\s?\d{1,2}월\s?\d{1,2}일\b',     # 21년 12월 30일
            r'\b\d{4}년\s?\d{1,2}월\b',                # 2021년 12월 (년월)
            r'\b\d{4}년\b',                            # 2021년
        ]
        for pat in date_patterns:
            text = re.sub(pat, "", text)

        # 전화번호 제거 패턴 추가
        phone_patterns = [
            r'\b01[016789][ -]?\d{3,4}[ -]?\d{4}\b',        # 010-1234-5678, 011 222 3333, 01612345678 등
            r'\b\d{2,4}[ -]?\d{3,4}[ -]?\d{4}\b',           # 02-123-4567, 053 123 4567, 0311234567 등 일반 번호
            r'\b\d{4}[ -]?\d{4}\b',                         # 1234-5678, 12345678 등
        ]
        for pat in phone_patterns:
            text = re.sub(pat, "", text)

        # 금액 패턴 제거
        price_patterns = [
            r'\b\d+원\b',                    # 1000원, 50000원
            r'\b\d+,\d+원\b',               # 1,000원, 50,000원
            r'\b\d+\.\d+원\b',              # 1000.5원
            r'\b\d+만원\b',                 # 1만원, 10만원
            r'\b\d+천원\b',                 # 1천원, 5천원
            r'\b\d+억원\b',                 # 1억원
            r'\$\d+',                       # $100, $50
            r'\b\d+달러\b',                 # 100달러
        ]
        for pat in price_patterns:
            text = re.sub(pat, "", text)

        # 시간 패턴 제거
        time_patterns = [
            r'\b\d{1,2}:\d{2}(?::\d{2})?\b',  # 14:30, 14:30:25
            r'\b\d{1,2}시\s?\d{1,2}분\b',     # 2시 30분, 2시30분
            r'\b\d{1,2}시간\b',               # 2시간, 10시간
            r'\b\d{1,2}분\b',                 # 30분, 5분
            r'\b\d{1,2}초\b',                 # 30초, 5초
            r'\b오전\s?\d{1,2}시\b',          # 오전 9시
            r'\b오후\s?\d{1,2}시\b',          # 오후 2시
        ]  
        for pat in time_patterns:
            text = re.sub(pat, "", text)

        # 영화 관련 패턴 제거
        movie_patterns = [
            r'\b\d+편\b',                    # 1편, 2편, 3편
            r'\b\d+부작\b',                  # 1부작, 2부작
            r'\b\d+기\b',                    # 1기, 2기
            r'\b\d+회차\b',                  # 1회차, 2회차
            r'\b\d+화\b',                    # 1화, 2화
            r'\b\d+분\s?\d+초\b',            # 120분 30초
            r'\b\d+분\b',                    # 120분 (영화 상영시간)
            r'\b\d+등급\b',                  # 15등급, 18등급
            r'\b\d+세\s?이상\b',             # 15세 이상
        ]    
        for pat in movie_patterns:
            text = re.sub(pat, "", text)

        # SNS/플랫폼 패턴 제거
        sns_patterns = [
            r'\b#\w+\b',                     # 해시태그 #영화 #추천
            r'\b@\w+\b',                     # 멘션 @username
            r'\bRT\b',                       # 리트윗 표시
            r'\b좋아요\s?\d+\b',             # 좋아요 100
            r'\b댓글\s?\d+\b',               # 댓글 50
            r'\b공유\s?\d+\b',               # 공유 20
            r'\b조회수\s?\d+\b',             # 조회수 1000
            r'\b구독자\s?\d+\b',             # 구독자 5000
        ]
        for pat in sns_patterns:
            text = re.sub(pat, "", text)

        # 기타 노이즈 패턴 제거
        noise_patterns = [
            r'\b\d+번\b',                    # 1번, 2번
            r'\b\d+개\b',                    # 1개, 2개
            r'\b\d+명\b',                    # 1명, 2명
            r'\b\d+장\b',                    # 1장, 2장
            r'\b\d+회\b',                    # 1회, 2회
            r'\b\d+차\b',                    # 1차, 2차
            r'\b\d+번째\b',                  # 1번째, 2번째
            r'\b\d+위\b',                    # 1위, 2위
            r'\b\d+등\b',                    # 1등, 2등
            r'\b\d+점\b',                    # 1점, 2점
            r'\b\d+점대\b',                  # 1점대, 2점대
            r'\b\d+점만점\b',                # 10점만점
            r'\b\d+점\s?만점\b',             # 10점 만점
        ]
        for pat in noise_patterns:
            text = re.sub(pat, "", text)

        # 특수 문자 및 기호 제거
        special_chars = [
            r'[★☆♥♡♠♣♦]',                  # 특수 기호
            r'[♪♫♬♩]',                      # 음악 기호
            r'[→←↑↓]',                      # 화살표
            r'[①②③④⑤⑥⑦⑧⑨⑩]',            # 원 숫자
            r'[⑴⑵⑶⑷⑸⑹⑺⑻⑼⑽]',            # 괄호 숫자
            r'[❶❷❸❹❺❻❼❽❾❿]',            # 검은 원 숫자
            r'[ⓐⓑⓒⓓⓔⓕⓖⓗⓘⓙ]',            # 원 문자
        ]
        for pat in special_chars:
            text = re.sub(pat, "", text)

        return text.strip()


    def _clean_text(self, text):
        """
        한국어 텍스트를 위한 기본 텍스트 정리 함수

        전처리 단계:
        1. 불완전한 한글 제거 (자음/모음만 있는 경우)
        2. 반복되는 감정 표현 정규화 (ㅋㅋㅋ, ㅠㅠㅠ 등)
        3. 과도한 문자 반복 축소 (4번 이상 → 3번으로)
        4. 특수문자 제거 (한글, 숫자, 기본 구두점, 감정표현 제외)
        5. 공백 정규화
        """
        if pd.isna(text):
            return ""

        text = str(text).strip()

        # 한국어 특화 전처리
        text = re.sub(
            r"[ㄱ-ㅎㅏ-ㅣ]+", "", text
        )  # 불완전한 한글 제거 (자음/모음만 있는 경우)
        text = re.sub(r"([ㅋㅎ])\1{2,}", r"\1\1", text)  # 웃음 표현 정규화: ㅋㅋㅋ+ → ㅋㅋ
        text = re.sub(
            r"([ㅠㅜㅡ])\1{2,}", r"\1\1", text
        )  # 슬픔 표현 정규화: ㅠㅠㅠ+ → ㅠㅠ
        text = re.sub(
            r"(.)\1{3,}", r"\1\1\1", text
        )  # 과도한 반복 축소: 4번 이상 반복 → 3번으로
        text = re.sub(r"[^\w\s가-힣.,!?ㅋㅎㅠㅜㅡ~\-]", " ", text)  # 필요한 문자만 유지
        text = re.sub(r"\s+", " ", text)  # 다중 공백을 단일 공백으로

        return text.strip()

    def fit(self, texts, labels=None):
        """학습 데이터로부터 전처리 정보 학습"""
        print("학습 데이터 기반 전처리 정보 수집 중...")

        # NOTE: 학습 데이터를 분석하여 전처리에 필요한 정보를 수집하고 저장하는 코드를 직접 구현해서 추가해보세요!
        # 예시:
        # - 도메인 특화 패턴 분석 (영화 리뷰 특성)
        # - 빈도 기반 노이즈 패턴 식별
        # - 라벨별 텍스트 특성 분석
        # - 어휘 사전 구축
        # - 정규화 규칙 최적화

        self.is_fitted = True
        print("✓ 전처리 파이프라인 학습 완료")

    def transform(self, texts):
        """전처리 적용"""
        if not self.is_fitted:
            print(
                "Warning: 파이프라인이 학습되지 않았습니다. 기본 전처리만 적용합니다."
            )

        return self.advanced_preprocess(texts)

    def fit_transform(self, texts, labels=None):
        """학습과 변환을 동시에 수행"""
        self.fit(texts, labels)
        return self.transform(texts)